# ouroboros-gx — Data Quality Runner

Run YAML data-contract checks against Fabric Lakehouse Delta tables using **Great Expectations Core**.

---

## Steps
1. Install / attach the `ouroboros-gx` library  
2. Configure run parameters  
3. Run a single contract  
4. *(Optional)* Batch-validate all contracts in a folder  
5. *(Optional)* Persist results to a Delta table

## 1 — Install dependencies

Skip this cell if `ouroboros-gx` is already attached as a Fabric Environment library.

In [ ]:
# Install from the wheel uploaded to your Fabric workspace files, or
# install directly from source during development.
%pip install great-expectations>=1.0.0 pydantic>=2.0.0 pyyaml

# If running from the repo root inside Fabric:
# %pip install -e /lakehouse/default/Files/ouroboros-gx

## 2 — Configuration

In [ ]:
# ------------------------------------------------------------------ #
# Paths and runtime variables — edit these for your environment
# ------------------------------------------------------------------ #

# Folder that contains your .yaml contract files
CONTRACTS_DIR = "/lakehouse/default/Files/dataquality/contracts"

# Fabric Lakehouse name (leave None to use the default attached Lakehouse)
LAKEHOUSE_NAME = None  # e.g. "my_lakehouse"

# Runtime variables referenced as ${VAR} inside contracts
RUNTIME_VARIABLES = {
    "FILTER_START_TIME": "2024-01-01 00:00:00",
}

# Delta table to persist DQ results (set to None to skip persistence)
RESULTS_TABLE = None  # e.g. "dq_results"

## 3 — Run a single contract

In [ ]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/ouroboros-gx/src")

from ouroboros_gx import DQRunner
from ouroboros_gx.result_reporter import display_results, summarize

runner = DQRunner(
    contract_path=f"{CONTRACTS_DIR}/dim_customer.yaml",
    variables=RUNTIME_VARIABLES,
    lakehouse_name=LAKEHOUSE_NAME,
)

result = runner.run()
display_results(result, dataset_name="dim_customer")

### 3b — Pass a DataFrame directly (optional)

Use this when you want to validate a DataFrame that has already been transformed in the notebook.

In [ ]:
# df = spark.table("dim_customer").filter("active = true")
# result = runner.run(dataframe=df)
# display_results(result, dataset_name="dim_customer")

## 4 — Batch-validate all contracts in a folder

In [ ]:
from pathlib import Path

all_summaries = []

for contract_file in sorted(Path(CONTRACTS_DIR).glob("*.yaml")):
    print(f"\n▶  Validating {contract_file.name} …")
    try:
        runner = DQRunner(
            contract_path=contract_file,
            variables=RUNTIME_VARIABLES,
            lakehouse_name=LAKEHOUSE_NAME,
        )
        res = runner.run()
        display_results(res, dataset_name=contract_file.stem)
        all_summaries.append(summarize(res, dataset_name=contract_file.stem))
    except Exception as exc:
        print(f"  ERROR: {exc}")
        all_summaries.append(
            {"dataset": contract_file.stem, "overall_success": False, "error": str(exc)}
        )

print("\n" + "=" * 62)
print("  Batch Summary")
print("=" * 62)
import pandas as pd
from IPython.display import display
display(pd.DataFrame(all_summaries))

## 5 — Persist results to Delta (optional)

In [ ]:
if RESULTS_TABLE and all_summaries:
    from datetime import datetime, timezone

    for s in all_summaries:
        s["run_timestamp"] = datetime.now(tz=timezone.utc).isoformat()

    results_df = spark.createDataFrame(pd.DataFrame(all_summaries))
    (
        results_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(RESULTS_TABLE)
    )
    print(f"Results written to Delta table: {RESULTS_TABLE}")
else:
    print("Persistence skipped (RESULTS_TABLE is None or no summaries available).")